## Section 1: Introduction to Google Earth Engine (5 min)

### What is GEE?
Google Earth Engine is a cloud-based platform for:
- Planetary-scale geospatial analysis
- Access to petabytes of satellite imagery
- Server-side processing (no local downloads needed for exploration)
- Free for research and education

### GEE Data Catalog
- **Sentinel-2:** 10m resolution multispectral imagery (13 bands)
- **Landsat:** 30m resolution, 50+ years of data
- **MODIS:** Daily global coverage
- **Climate, Terrain, Land Cover:** Pre-processed datasets

### Why Sentinel-2?
- **High Resolution:** 10m for RGB + NIR
- **Revisit Time:** 5 days (with both satellites)
- **Free & Open:** Copernicus program
- **Spectral Bands:** Ideal for vegetation, water, urban analysis

### Sentinel-2 Bands (Key Bands for ML)
| Band | Name | Wavelength (μm) | Resolution (m) | Use Case |
|------|------|-----------------|----------------|----------|
| B2 | Blue | 0.49 | 10 | Water, atmosphere |
| B3 | Green | 0.56 | 10 | Vegetation vigor |
| B4 | Red | 0.665 | 10 | Chlorophyll absorption |
| B8 | NIR | 0.842 | 10 | Vegetation structure |
| B11 | SWIR1 | 1.61 | 20 | Moisture, fire |
| B12 | SWIR2 | 2.19 | 20 | Geology, soil |

## Section 2: GEE Setup and Authentication (5 min)

### Step 1: Install Earth Engine API

In [ ]:
# Install required packages (run once)
!pip install earthengine-api geemap folium geopandas

### Step 2: Authenticate GEE
You'll need to authenticate once per environment:

In [ ]:
import ee

# Trigger authentication (will open browser)
try:
    ee.Initialize()
    print("✅ Already authenticated!")
except:
    print("🔐 Authentication required...")
    ee.Authenticate()
    ee.Initialize()
    print("✅ Authentication successful!")

**Authentication Steps:**
1. Click the link that appears
2. Sign in with your Google account
3. Grant Earth Engine permissions
4. Copy the authorization code
5. Paste it back into the notebook

In [ ]:
# Import additional libraries
import geemap
import folium
import datetime
import pandas as pd
from pathlib import Path

print("✅ All libraries loaded successfully!")

## Section 3: Define Area of Interest (AOI) in Iceland (8 min)

### Choose Your Study Area
For this course, we'll focus on areas in Iceland with diverse land cover:
- **Reykjavik Region:** Urban + coastal
- **Þingvellir:** Vegetation + volcanic
- **Vatnajökull:** Glaciers + bare rock

Let's define a study area near **Þingvellir National Park**:

In [ ]:
# Define AOI as a bounding box (lon/lat)
# Þingvellir area: [west, south, east, north]
aoi_coords = [-21.3, 64.2, -21.0, 64.4]

# Create GEE geometry
aoi = ee.Geometry.Rectangle(aoi_coords)

print(f"AOI Center: {aoi.centroid().coordinates().getInfo()}")
print(f"AOI Area: {aoi.area().divide(1e6).getInfo():.2f} km²")

In [ ]:
# Visualize AOI on interactive map
Map = geemap.Map(center=[64.3, -21.15], zoom=10)
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map.add_basemap('SATELLITE')
Map

### Alternative: Draw Your Own AOI
You can also draw directly on the map:

In [ ]:
# Interactive AOI drawing
Map = geemap.Map(center=[64.3, -21.15], zoom=10)
Map.add_basemap('SATELLITE')

# Instructions:
print("📍 Use the drawing tools to define your AOI:")
print("   1. Click the rectangle/polygon tool on the left")
print("   2. Draw your area of interest")
print("   3. Run the next cell to extract coordinates")

Map

In [ ]:
# Extract drawn AOI (if you used drawing tools)
# aoi = Map.draw_last_feature.geometry()
# print(f"Custom AOI coordinates: {aoi.bounds().getInfo()['coordinates']}")

## Section 4: Query Sentinel-2 Image Collection (12 min)

### Define Search Parameters

In [ ]:
# Date range: Summer 2024 (less snow, more vegetation)
start_date = '2024-06-01'
end_date = '2024-09-30'

# Cloud cover threshold
max_cloud_cover = 20  # percent

print(f"🔍 Searching for Sentinel-2 scenes:")
print(f"   Date Range: {start_date} to {end_date}")
print(f"   Max Cloud Cover: {max_cloud_cover}%")
print(f"   AOI: {aoi.area().divide(1e6).getInfo():.2f} km²")

### Query Image Collection

In [ ]:
# Query Sentinel-2 Surface Reflectance (Level 2A)
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(aoi)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_cover)))

# Get collection size
count = collection.size().getInfo()
print(f"\n✅ Found {count} scenes matching criteria")

### Inspect Scene Metadata

In [ ]:
# Extract metadata for all scenes
def extract_metadata(image):
    return ee.Feature(None, {
        'scene_id': image.get('PRODUCT_ID'),
        'date': image.date().format('YYYY-MM-dd'),
        'cloud_cover': image.get('CLOUDY_PIXEL_PERCENTAGE'),
        'solar_azimuth': image.get('MEAN_SOLAR_AZIMUTH_ANGLE'),
        'solar_zenith': image.get('MEAN_SOLAR_ZENITH_ANGLE')
    })

metadata = collection.map(extract_metadata).getInfo()['features']
df_scenes = pd.DataFrame([f['properties'] for f in metadata])

print("\n📊 Available Scenes:")
print(df_scenes.to_string(index=False))

In [ ]:
# Sort by cloud cover and select best scenes
df_scenes_sorted = df_scenes.sort_values('cloud_cover')
print("\n🌤️ Top 5 Clearest Scenes:")
print(df_scenes_sorted.head().to_string(index=False))

### Visualize Best Scene

In [ ]:
# Get the clearest image
best_image = ee.Image(collection.sort('CLOUDY_PIXEL_PERCENTAGE').first())

# Visualization parameters for True Color (RGB)
vis_params_rgb = {
    'min': 0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue
    'gamma': 1.4
}

# Visualization for False Color (NIR, Red, Green) - highlights vegetation
vis_params_nir = {
    'min': 0,
    'max': 3000,
    'bands': ['B8', 'B4', 'B3'],  # NIR, Red, Green
    'gamma': 1.4
}

# Create map with both visualizations
Map = geemap.Map(center=[64.3, -21.15], zoom=11)
Map.addLayer(best_image, vis_params_rgb, 'True Color (RGB)')
Map.addLayer(best_image, vis_params_nir, 'False Color (NIR)', shown=False)
Map.addLayer(aoi, {'color': 'red'}, 'AOI', opacity=0.3)

# Add scene info
scene_date = best_image.date().format('YYYY-MM-dd').getInfo()
cloud_pct = best_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
print(f"\n🖼️ Displaying Best Scene:")
print(f"   Date: {scene_date}")
print(f"   Cloud Cover: {cloud_pct:.1f}%")

Map

## Section 5: Select and Export Scenes (10 min)

### Select Multiple Scenes for Training
For ML, we want temporal diversity:

In [ ]:
# Select 4 scenes across the season
num_scenes = 4
selected_collection = collection.sort('CLOUDY_PIXEL_PERCENTAGE').limit(num_scenes)

# Get dates of selected scenes
selected_dates = selected_collection.aggregate_array('system:time_start').getInfo()
selected_dates = [datetime.datetime.fromtimestamp(d/1000).strftime('%Y-%m-%d') for d in selected_dates]

print(f"\n✅ Selected {num_scenes} scenes for dataset:")
for i, date in enumerate(selected_dates, 1):
    print(f"   {i}. {date}")

### Prepare Export Configuration
We'll export pre-processed imagery for the ML pipeline:

In [ ]:
# Select bands for ML (10m and 20m resampled to 10m)
ml_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']  # RGB + NIR + SWIR

# Cloud masking function
def mask_clouds(image):
    """Mask clouds using SCL band (Scene Classification Layer)"""
    scl = image.select('SCL')
    # Keep clear (4,5,6) and exclude clouds (8,9,10)
    mask = scl.neq(8).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(3))
    return image.updateMask(mask).select(ml_bands)

# Apply cloud masking
masked_collection = selected_collection.map(mask_clouds)

print("✅ Applied cloud masking to selected scenes")

### Export to Google Drive (Alternative: Local Download)

For small AOIs, we can download directly. For larger areas, export to Google Drive:

In [ ]:
# Method 1: Export to Google Drive (for larger areas)
# Note: This triggers an export task, not immediate download

def export_to_drive(image, description):
    """Export single image to Google Drive"""
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder='iceland_ml_course',
        region=aoi,
        scale=10,  # 10m resolution
        crs='EPSG:4326',
        maxPixels=1e9
    )
    task.start()
    return task

# Export each scene
print("🚀 Starting export tasks...\n")
tasks = []
for i, date in enumerate(selected_dates, 1):
    img = ee.Image(masked_collection.toList(num_scenes).get(i-1))
    task = export_to_drive(img, f'sentinel2_iceland_{date.replace("-", "")}')
    tasks.append(task)
    print(f"   ✓ Task {i} started: {date}")

print(f"\n📌 Monitor progress at: https://code.earthengine.google.com/tasks")
print(f"📂 Files will appear in Google Drive: iceland_ml_course/")

In [ ]:
# Method 2: Direct download for small AOI (faster for this lab)
import requests
import os

# Save to project directory
output_dir = Path(os.getenv('PROJECT_training2600')) / 'my_workspace' / 'data' / 'sentinel2'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"💾 Downloading scenes to: {output_dir}\n")

for i, date in enumerate(selected_dates[:2], 1):  # Download first 2 for speed
    img = ee.Image(masked_collection.toList(num_scenes).get(i-1))
    
    # Get download URL
    url = img.getDownloadURL({
        'region': aoi,
        'scale': 10,
        'format': 'GEO_TIFF'
    })
    
    # Download
    filename = output_dir / f'sentinel2_{date.replace("-", "")}.tif'
    response = requests.get(url)
    with open(filename, 'wb') as f:
        f.write(response.content)
    
    print(f"   ✓ Downloaded: {filename.name}")

print(f"\n✅ Download complete!")

### Save Scene Metadata

In [ ]:
# Save metadata for reference
metadata_file = output_dir / 'scene_metadata.csv'
df_scenes_sorted.head(num_scenes).to_csv(metadata_file, index=False)

print(f"📝 Saved metadata to: {metadata_file}")
print(f"\n{df_scenes_sorted.head(num_scenes).to_string(index=False)}")

## Summary & Next Steps

### What We Covered
✅ Set up Google Earth Engine authentication  
✅ Defined an AOI in Iceland  
✅ Queried Sentinel-2 imagery with filters  
✅ Visualized and selected optimal scenes  
✅ Exported/downloaded imagery for ML pipeline  

### Data Acquired
- **Scenes:** 4 Sentinel-2 images (Summer 2024)
- **Bands:** B2, B3, B4, B8, B11, B12 (6 bands)
- **Resolution:** 10m
- **Format:** GeoTIFF
- **Cloud Cover:** < 20%

### Key GEE Concepts
- **ImageCollection:** Time series of satellite images
- **Filtering:** Spatial, temporal, and attribute filters
- **Cloud Masking:** Remove cloudy pixels
- **Export:** Server-side processing for large areas

### Prepare for Lab 4
Next lab: **Data Preprocessing & Patch Extraction**
- We'll load the downloaded imagery
- Extract patches for training
- Apply normalization
- Match with CORINE land cover labels

### Additional Resources
- **GEE Catalog:** https://developers.google.com/earth-engine/datasets
- **Sentinel-2 Bands:** https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-2-msi/resolutions/radiometric
- **geemap Tutorials:** https://geemap.org/tutorials

### Homework (Optional)
1. Try different AOIs in Iceland (coast, urban, volcanic)
2. Experiment with different date ranges
3. Compare cloud-free scenes across seasons
4. Calculate vegetation indices (NDVI, EVI)

---

**Excellent work!** You've successfully acquired real satellite data! 🛰️